In [ ]:

import kagglehub

# Download latest version
path = kagglehub.dataset_download("reflex7/cxr-data-set")

print("Path to dataset files:", path)

100%|██████████| 16.9G/16.9G [17:09<00:00, 17.6MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/reflex7/cxr-data-set/versions/1


In [ ]:
import os

cgan_file_path = '/content/cgan.py'

# Read the content of cgan.py
with open(cgan_file_path, 'r') as f:
    cgan_content = f.read()

# Replace the incorrect import statements
modified_cgan_content = cgan_content.replace(
    'from models.encoder import Encoder',
    'from encoder import Encoder'
)
modified_cgan_content = modified_cgan_content.replace(
    'from models.classifier import Classifier',
    'from classifier import Classifier'
)

# Write the modified content back to cgan.py
with open(cgan_file_path, 'w') as f:
    f.write(modified_cgan_content)

print("cgan.py updated successfully. Please re-run the cell with the imports (tFzpuasVOJ1A).")

FileNotFoundError: [Errno 2] No such file or directory: '/content/cgan.py'

In [ ]:
import sys
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
sys.path.insert(0, '/content')
from encoder import Encoder
from cgan import Generator, Discriminator,disc_loss, gen_loss, weights_init, Classifier
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch.optim as optim
import os
transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
n_critics = 5
dataset = datasets.ImageFolder(
    root="/root/.cache/kagglehub/datasets/reflex7/cxr-data-set/versions/1",
    transform=transform
)
loader = DataLoader(dataset, batch_size=32, shuffle=True,num_workers=2,pin_memory=True,persistent_workers=True)
gen = Generator().to("cuda")
disc = Discriminator(10).to("cuda")
encoder = Encoder().to("cuda")
classifier = Classifier().to("cuda")
classifier.apply(weights_init)
optimizer_gen  = optim.Adam(gen.parameters(),lr = 2e-4, betas=(0.0,0.9))
optimizer_disc  = optim.Adam(disc.parameters(),lr = 2e-4, betas=(0.0,0.9))
print(gen)
print(disc)

In [ ]:
epochs = 50
for epoch in range(epochs):
    image_totest, labels_totest = None, None
    batch_loss_gen, batch_loss_disc = [], []

    for batch_idx, (image, labels) in enumerate(loader):
        image  = image.to("cuda")
        labels = labels.to("cuda")

        for _ in range(n_critics):
            optimizer_disc.zero_grad()
            loss_d = disc_loss(image, labels, gen, disc, encoder)
            loss_d.backward()
            optimizer_disc.step()
        optimizer_gen.zero_grad()
        target_labels = 1 - labels
        target_cls    = target_labels.unsqueeze(1).float()
        loss_g = gen_loss(image, labels, target_labels, target_cls,
                          gen, disc, classifier, encoder)
        loss_g.backward()
        optimizer_gen.step()

        if batch_idx == 0:
            image_totest  = image
            labels_totest = labels

        batch_loss_gen.append(loss_g.item())
        batch_loss_disc.append(loss_d.item())

    with torch.no_grad():
        z = encoder(image_totest)
        target_labels_viz = 1 - labels_totest

        source_imgs = gen(z, labels_totest).detach().cpu()
        counter_imgs = gen(z, target_labels_viz).detach().cpu()

        source_imgs  = (source_imgs  + 1) / 2
        counter_imgs = (counter_imgs + 1) / 2
        original_cpu = (image_totest.cpu() + 1) / 2
        grid = make_grid(
            torch.cat([original_cpu, source_imgs, counter_imgs], dim=0),
            nrow=image_totest.size(0)
        )
        plt.figure(figsize=(12, 4))
        plt.imshow(grid.permute(1, 2, 0))
        plt.title(f"Epoch {epoch+1} | Top: Original  Mid: Recon  Bot: Counterfactual")
        plt.axis("off")
        plt.show()